In [110]:
#piszemy narzędzie które dostaje daną firmę, opis czym się zajmuje w stringu i otrzymuje 2 narzędzia 1 które pisze mail i zapisuje do csv 2 które 
# zapisuje odrzucony mail z powodem odrzucenia  

#Tools:
#save_email(company, email_content, subject) — zapisuje zatwierdzony mail do CSV
#flag_generic_email(company, reason) — zapisuje odrzucony mail z powodem żebyś wiedział co poprawić

In [111]:

# import

from dotenv import load_dotenv
from pydantic import BaseModel
from openai import OpenAI
import json
import os
import csv


openai = OpenAI()

In [112]:

# piszemy 1 narzędzie czyli funkcje która ma zawierać zmienne company, email i subject po to by napisać mail i wrzucić go do csv


def save_email(company,email_content,subject): 

    # za pomocą metody open with tworzymy obiekt f który moze nadpisac plik csv, metodą append 

    with open('mail.csv' , 'a' , newline='' , encoding='utf-8') as f:

        # do zmiennej mail przekazujemy obiekt csv z metodą write i przekazujemy obiekt f jako parametr

        mail = csv.writer(f)

        mail.writerow([company,email_content,subject])

    return {'recorded' : 'ok'}



In [113]:

# teraz piuszemy 2 narzędzie czyli funkcje która ma zawierać nazwe firmy czyli company i informacje dalczego mail został odrzucony czyli reason

def repeat_mail(company,reason):

    # robiumy dokladnie to samo metodą with open tworzyny obiekt f który ma dany csv metode append

    with open('repeatmail.csv' , 'a' , newline="", encoding='utf-8') as f:

        # tworzymy zmienną do kórej przekazujemy moduł csv, który pozwala pracować na plikach csv metoda writen a jako parametr przekazujemy obiekt f tworzony metodą with open

        mail = csv.writer(f)

        mail.writerow([company,reason])


    return {'recorded' : 'ok'}


In [114]:
# teraz piszemy json tak jak wymaga tego api, to dzieki niemu tools dziala 
# kazdy wymaga name description parametrs  ktory ma type object i properties które zaczyna sie od nazwy paramtetru który ma type i description, required które jest listą properties i additional

save_email_json = {

    'name' : "save_email",
    'description' : 'zapsisuje mail do pliku csv ',
    # dodajemy parametrs czyli parametry funkcji narzędzia
    "parameters" : {
        'type' : 'object',
        'properties' : {
            'company' : {
                'type' : 'string',
                "description" : 'zapisuje nazwę firmy'
            },
            'email_content' : {
                'type' : 'string',
                'description' : 'zapisuje treść maila'
            },
            'subject' : {
                'type' : 'string',
                'description': 'zapisuje temat maila'
            }
        },
        'required' : ['company',"email_content",'subject'],
        "additionalProperties": False

    }
       

}

In [115]:
# 2 tools

repeat_mail_json = {
    'name' : 'repeat_mail',
    'description' : 'zapisuje mail do pliku csv z nazwą firmy i powodem odrzutu maila',
    'parameters' : {

        'type' : 'object',
        'properties' : {
            'company' : {

                'type' : 'string',
                'description': 'zapisuje nazwe firmy'

            },
            'reason' : {
                'type' : 'string',
                'description' : 'podaje powód przez który treść maila zostałą odrzucona'
            },

        },
         'required' : ['company','reason'],
         "additionalProperties": False


    }
}

In [116]:
# piszemy obiekt tools

tools = [
    {'type' : 'function' , "function" : save_email_json},

    {'type' : 'function' , 'function' : repeat_mail_json}

]

In [117]:
# piszemy funkcje która pozwoli narzedziu skorzystać z danego tools 
# jako parametr przekazujemy tool_calls 



def hande_tool_calls(tool_calls): 

    # tworzymy tablice results gdzie bedziemy wrzucać wyniki tej funkcji, czyli uzyte narzędzia wraz zargumentami

    results = []

    # zapętlamy tool_calls zeby kod wywolywal sie co iteracje na innym narzędziu

    for tool_call in tool_calls:

        # wpierw pobieramy nazwę tool
        name = tool_call.function.name 

        # teraz pobieramy argumenty z json danego tool metodą loads

        arguments = json.loads(tool_call.function.arguments)

        # teraz robimy funkcje sterującą if else i i przekazujemy dane narzędzie, nazwe funkjci i argumenty do results jesli name jest zgodny z danym narzedziem musimy przekazać nazwę funkcji

        if name == 'save_email':
            
            result = save_email(**arguments)

        elif name == 'repeat_mail':

            result = repeat_mail(**arguments)


    # teraz do tablicy results musimy przekazac rzeczy role narzędzia czyli tools content czyli wartosc resuklts = wynik daej funkcju która zostalaurzyta wraz z argumentami zamieniona prrez json.dumps i id anego narzędzia

    results.append({
        'role': 'tool',
        'content' : json.dumps(result),
        'tool_call_id' : tool_call.id

    }) 
    return results 

In [118]:
system_prompt = """Jesteś asystentem do generowania maili outreachowych dla Last Agency - polskiej agencji SEO/SEM/GEO/AI Search z Poznania.

Dostajesz nazwę firmy, branżę i krótki opis czym się zajmują. Twoim zadaniem jest napisanie spersonalizowanego maila partnerskiego do tej firmy.

Jak działasz:
1. Na podstawie podanych informacji napisz spersonalizowanego maila partnerskiego
2. Mail musi być po polsku, naturalny i konkretny
3. Jeśli mail jest dobry - użyj narzędzia save_email żeby go zapisać
4. Jeśli mail jest zbyt ogólny lub szablonowy - użyj repeat_mail z powodem

Dobry mail:
- Odnosi się do konkretnej branży i usług firmy
- Ma jasny hook - dlaczego akurat ta firma miałaby być partnerem Last Agency
- Nie brzmi jak masowy szablon
- Jest krótki, konkretny i po polsku
- Proponuje konkretny model współpracy - white-label lub referral"""


In [119]:
evaluator_system_prompt = """Oceniasz czy wygenerowany mail outreachowy jest dobry i nadaje się do wysłania przez Last Agency - polską agencję SEO/SEM/GEO/AI Search z Poznania.

Dobry mail:
- Jest po polsku i ma naturalny ton
- Odnosi się do konkretnej branży i usług firmy do której jest wysyłany
- Ma jasny hook - konkretny powód dlaczego ta firma miałaby być partnerem Last Agency
- Nie brzmi jak masowy szablon
- Proponuje konkretny model współpracy - white-label lub referral
- Jest krótki i konkretny

Odrzuć mail jeśli:
- Brzmi jak szablon który można wysłać do każdej firmy
- Nie odnosi się do konkretnych usług lub branży firmy
- Jest po angielsku
- Nie ma konkretnego hooka

Zwróć TYLKO JSON z polami:
- is_acceptable (bool): czy mail nadaje się do wysłania
- feedback (string): co poprawić jeśli mail jest odrzucony"""


In [120]:
# budujemy Clase Evaluaion na pdostawie Basemodel od pydanitca, tu trafia wybnik funkcji z modelu evalaution

class Evaluation(BaseModel):
    is_acceptable : bool
    feedback : str

In [121]:
# budujemy model evalute odpalany w funkcji chat ma zazadanie sprawdzic odpowiedz 1 modelu i zweryfikac ją dzieli evalautor system prompt i przesłąc do Evaluation

def evaluate(reply,message,history) -> Evaluation:

    # budujemy liste message
    messages = [{'role' : 'system' , 'content' : evaluator_system_prompt}] + [{'role' : 'user', 'content' : message} ] 

    # budujemy model i jego odpowiedz ale musimy zwrócić obietk metodą beta parse i dac format odpowiedzi

    response = openai.beta.chat.completions.parse(

        model = 'gpt-4.1-mini',
        messages=messages,
        response_format= Evaluation
    )

    return response.choices[0].message.parsed



In [122]:
# budujemy model rerun który odpala evalaute jesli 1 model dał bledana odpowiedz 
# model dziala w petli bo ma dostep do narzędzi

# odpala funkcje handle 

def rerun(reply,message,history,feedback): 

    # budujemy nowy prompt
    new_system_prompt = system_prompt + f'poprzednia wiadomosc zostala odrzucona'
    new_system_prompt += f'tutaj masz wiadomość która została odrzucona {reply}'
    new_system_prompt += f'tutaj masz pytanie usera {message}'
    new_system_prompt += f'tutaj masz feedback od evaluaotra dlaczego została odrzucona ta odpowiedź {feedback}'  


    # lista messages 

    messages = [{'role' :'system', "content" : new_system_prompt }] + history + [{'role' : 'user' , 'content' : message}]

    # teraz bedziemy zapętlać model zeby mógl co wywołanie uzyc narzędzia dopóki chce go uzyc

    done = False

    while not done: 

        # piszemy model modelowi dostaraczamy liste tools'g

        response = openai.chat.completions.create(

            model = 'gpt-4.1-mini',
            messages=messages,
            tools= tools
        )

        # teraz do zmiennej finish reason przekazujemy wlasnie te zmienna z obiektu ona ma 2 statny 1 to tool_calls czyli uzycie narzędzi, wówacas model moze działąć dalej
        # 2 to stop który konczy działanie, oznacza ze model wykorszytal kazde narzedzie jakie miał lub chciał uzyc

        finish_reason = response.choices[0].finish_reason 

        # teraz if elese w zaleznosci od wartosni finish_reason
        # jesli odpoiwedz zakonczyla sie wywołaniem narzędzi to musimy zrobic poare rzeczy
        if finish_reason == 'tool_calls':

            # wpierw zwracamy cały obket message, tam mamy info o danym narzędziu  1 pobieramy obiekt message gdzie znajduje sie całe tool_calls czyli funkcja narzedzia z argumentami
            message_obj = response.choices[0].message
            # teraz z obiektu wyciagamy nazwe funkji wraz z argumentami metodą tool_calls
            tool_calls = message_obj.tool_calls
            # teraz wywołujemy  funkcje habdle_tool_calls z argumentem tools calls czyli nazwe funkcji która odpala narzędzia wraz z arguemntami i przekazujemy je do funkcju kóra zwraca liste uzytych narzedzi do results
            result = hande_tool_calls(tool_calls)
            # teraz przekazujemy cały obiekt message_obj do listy message tam znajduje sie info o wszustkich tool_calls
            messages.append(message_obj)
            # teraz do message dodajemy wynik funkcji handle_tool_calls z argumentami które zostały uzyte czyli dodajemy konkretne narzedzie z argumentami
            messages.extend(result) 

        else: 
            result = True

    # zwracamy wynik modelu 
    return response.choices[0].message.content

In [123]:
# teraz piszemy głowna funkcje jktóra odpala wszystkio xD





def chat(message,history):

    # piszemy message
    messages =  [{'role' : 'system', 'content' : system_prompt}] + history + [{'role' : 'user'  ,'content'  : message }]

    # PISZEMY MODEl w petli zeby tez mogl korzystac z tools

    done = False

    while not done:

        response = openai.chat.completions.create(

            model = 'gpt-4.1-mini',
            messages=messages,
            tools=tools
        )

        # do zmieinnej przekazujemy wynik zakonczenia dzialania modelu
        finish_reason = response.choices[0].finish_reason

        # funkcja sterujaca w zaleznosci od wartosi finih reason czyli odpwiediz modelu . albo uzycie narzedzi tool_calls albo stop

        if finish_reason == 'tool_calls':
            # jesli model uzyl narzedzi to

            # pobieramy cały obket message z odpoweidzi modelu tamm mamy m.nnymi tool_calls czyli nazwe uzytej funkci i jej argumenty
            message_obj = response.choices[0].message

            # pobieramy konkretne narzedzia uzyte przez model, nazwa funkcji i argumenty

            tool_calls = message_obj.tool_calls

            # do result przekazujemy wynik funkcji handle która otrzymuje odpowidi tool_calls jako parametr

            result = hande_tool_calls(tool_calls)

            # do messaage appendujemy cały obiekt message 

            messages.append(message_obj)

            # do message extendujemy resut czyli wynik funkcji handle tool calls z parametrem tool_calls

            messages.extend(result)

        else:
            done = True
#do zmiennej reply czyli odpowiedz modelu ktorej dalej potrzebuje evalautor lub rerun przekazujmy wynik zodpowiedzi modelu
    reply = response.choices[0].message.content

    # uruchamiamy model evalautor

    evaluation = evaluate(reply,message,history)

    # teraz if else w oparciu o Evalautin i jego atrybuty is_acceotable

    if evaluation.is_acceptable:
        print(reply)

    else:
        # jesli evalautaor is acceptable jest false to puruchamiamy rerun i zracamy odpowiedz modelu
        reply =  rerun(reply,message,history,Evaluation.feedback)
    return reply

    

In [124]:
history = []

msg1 = """Firma: Tebim Pro
Branża: agencja e-commerce
Opis: tworzą sklepy internetowe na PrestaShop dla polskich firm, specjalizują się w migracji sklepów i custom development, klienci to głównie średnie sklepy e-commerce w Polsce"""

result1 =  chat(msg1,history)


print(result1)

Cześć,

Zauważyliśmy, że specjalizujecie się w tworzeniu sklepów internetowych na PrestaShop oraz migracji i custom development dla polskich średnich sklepów e-commerce. Jako Last Agency, agencja SEO/SEM/GEO/AI Search z Poznania, chętnie nawiążemy z Wami współpracę, by wspólnie wspierać rozwój Waszych klientów poprzez skuteczne działania w zakresie pozycjonowania i marketingu internetowego.

Proponujemy model white-label, gdzie możemy działać jako Wasz „cichy” partner w obszarze SEO/SEM lub model referral, dzięki któremu wspólnie pozyskujemy nowych klientów i dzielimy się prowizją.

Chętnie porozmawiamy o szczegółach – dajcie znać, czy jesteście zainteresowani.

Pozdrawiam,  
[Twoje Imię] z Last Agency
Cześć,

Zauważyliśmy, że specjalizujecie się w tworzeniu sklepów internetowych na PrestaShop oraz migracji i custom development dla polskich średnich sklepów e-commerce. Jako Last Agency, agencja SEO/SEM/GEO/AI Search z Poznania, chętnie nawiążemy z Wami współpracę, by wspólnie wspierać 